# Packages

In [2]:
import warnings
warnings.filterwarnings("ignore")
import os
from pathlib import Path

import pandas as pd
import numpy as np
import polars as pl
import scipy.stats as stats
import seaborn as sns
import statsmodels.formula.api as smf
import statsmodels.api as sm
import matplotlib.pyplot as plt
import math
import plotly.express as px
import plotly

from statsmodels.tsa.deterministic import DeterministicProcess, CalendarFourier
from statsmodels.graphics.tsaplots import plot_pacf

from sklearn.model_selection import GroupKFold, KFold, train_test_split, cross_val_score, TimeSeriesSplit
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, r2_score, make_scorer, accuracy_score, median_absolute_error, classification_report, mean_absolute_error, roc_auc_score, roc_curve
from sklearn.pipeline import make_pipeline
from sklearn.decomposition import PCA
from imblearn.over_sampling import RandomOverSampler
from scipy.optimize import minimize

import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor, RandomForestClassifier
from sklearn.linear_model import Ridge
from sklearn.svm import LinearSVC
from sklearn.svm import SVR
from sklearn.linear_model import LogisticRegression

import optuna
import unicodedata


pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.float_format', lambda x: "%.4f" % x)
# pd.options.plotting.backend = "plotly"

plt.style.use('ggplot')
sns.set_style('darkgrid')

## Helper Functions

In [3]:
def get_info(df):
    missing_values_train = pd.DataFrame({'Feature': df.columns,
                              'No. of Missing Values': df.isnull().sum().values,
                              '% of Missing Values': ((df.isnull().sum().values)/len(df)*100)})

    unique_values = pd.DataFrame({'Feature': df.columns,
                                'No. of Unique Values': df.nunique().values})

    feature_types = pd.DataFrame({'Feature': df.columns,
                                'DataType': df.dtypes})

    merged_df = pd.merge(missing_values_train, unique_values, on='Feature', how='left')
    merged_df = pd.merge(merged_df, feature_types, on='Feature', how='left')

    return merged_df

def get_cols_as_list_by_type(df, target_col=None):
    if target_col is not None:
        df = df.drop(target_col, axis=1)
        
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    categorical_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    return numeric_cols, categorical_cols

# plot all pairs in rows of 6 
def pairplot_list_against_list(df, l1, l2):
    pairs = [(x, y) for x in l1 for y in l2]
    num_plots = len(pairs)
    cols = 6
    rows = math.ceil(num_plots / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten()

    for i, (x, y) in enumerate(pairs):
        sns.scatterplot(data=df, x=x, y=y, ax=axes[i])
        axes[i].set_xlabel(x)
        axes[i].set_ylabel(y)

    # Hide any unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    plt.show()

# Reading Data

In [4]:
data_dir = "./kyle"

data_dir_path = Path(data_dir)
score_file_paths = [
    f for f in data_dir_path.rglob("*.csv") 
    if f.is_file() and (f.name != "judge_nationalities.csv" and f.name != "judges_nationalities_v2.csv")
]
df_list = [pd.read_csv(f) for f in score_file_paths]
data_df = pd.concat(df_list, ignore_index=True)
judge_cols = [f"J{i}" for i in range(1,10)]
data_df = data_df.assign(judge_avg=lambda x: x[judge_cols].mean(axis=1))
data_df.head()

,rank,name,noc,starting_number,tss,tes,tpcs,deductions,base_value,competition,element,element_no,extra_points,factor,final_score,goe,info,is_element,program_component,year,J1,J2,J3,J4,J5,J6,J7,J8,J9,is_short_program,category,event_type,judge_avg
0,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,6.6000,wc2018,3Tw4,1.0000,0.0000,NaN,8.7000,2.1000,NaN,1,NaN,1718,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,1,pairs,individual,3.0000
1,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,4.4000,wc2018,3S,2.0000,0.0000,NaN,5.9000,1.5000,NaN,1,NaN,1718,2.0000,2.0000,3.0000,2.0000,3.0000,1.0000,1.0000,3.0000,2.0000,1,pairs,individual,2.1111
2,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,5.5000,wc2018,3FTh,3.0000,0.0000,NaN,6.4000,0.9000,NaN,1,NaN,1718,2.0000,0.0000,1.0000,2.0000,3.0000,1.0000,0.0000,2.0000,1.0000,1,pairs,individual,1.3333
3,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,4.5000,wc2018,3Li4,4.0000,0.0000,NaN,6.0000,1.5000,NaN,1,NaN,1718,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,2.0000,3.0000,3.0000,1,pairs,individual,2.8889
4,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,3.9000,wc2018,StSq4,5.0000,0.0000,NaN,6.0000,2.1000,NaN,1,NaN,1718,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,1,pairs,individual,3.0000


In [5]:
data_df.shape

(62049, 33)

## Adding Judge Nationalities

In [6]:
# def normalize_name(name):
#     if not isinstance(name, str):
#         return name

#     replacements = {
#         'ä': 'a', 'ö': 'o', 'ü': 'u',
#         'Ä': 'a', 'Ö': 'o', 'Ü': 'u',
#         'ø': 'o', 'Ø': 'o',
#         'ß': 'ss',
#         'é': 'e', 'è': 'e', 'ê': 'e', 'ë': 'e',
#         'É': 'e', 'È': 'e', 'Ê': 'e', 'Ë': 'e',
#         'á': 'a', 'à': 'a', 'â': 'a', 'ã': 'a', 'å': 'a',
#         'Á': 'a', 'À': 'a', 'Â': 'a', 'Ã': 'a', 'Å': 'a',
#         'í': 'i', 'ì': 'i', 'î': 'i', 'ï': 'i',
#         'Í': 'i', 'Ì': 'i', 'Î': 'i', 'Ï': 'i',
#         'ó': 'o', 'ò': 'o', 'ô': 'o', 'õ': 'o',
#         'Ó': 'o', 'Ò': 'o', 'Ô': 'o', 'Õ': 'o',
#         'ú': 'u', 'ù': 'u', 'û': 'u',
#         'Ú': 'u', 'Ù': 'u', 'Û': 'u',
#         'ñ': 'n', 'Ñ': 'n',
#         'ç': 'c', 'Ç': 'c',
#         'ý': 'y', 'ÿ': 'y', 'Ý': 'y', 'Ÿ': 'y'
#     }

#     for old, new in replacements.items():
#         name = name.replace(old, new)

#     return name.lower().strip()


# ------------------------
# Normalization
# ------------------------

def normalize_name(name):
    if not isinstance(name, str):
        return name
    
    # Remove accents safely
    name = unicodedata.normalize("NFKD", name)
    name = "".join(c for c in name if not unicodedata.combining(c))
    
    return name.strip()


def name_key(name):
    name = name.lower()
    if not isinstance(name, str):
        return name
    
    name = normalize_name(name)
    tokens = name.split()
    tokens = sorted(tokens)
    
    return " ".join(tokens)



In [7]:
judge_path = data_dir + "/judge_nationalities.csv"

judge_df = pd.read_csv(judge_path)
judge_df = judge_df.dropna(subset=["category"])

judge_dfv2 = pd.read_csv(data_dir + "/judges_nationalities_v2.csv")
judge_dfv2 = judge_dfv2.rename(
    columns={"iso3c": "judge_nationality", "Name": "judge_name"}
)

# Create order-invariant keys

judge_df["judge_name"] = judge_df["judge_name"].apply(name_key)
judge_dfv2["judge_name"] = judge_dfv2["judge_name"].apply(name_key)

# Build lookup using key
judge_lookup = (
    judge_dfv2[["judge_name", "judge_nationality"]]
    .drop_duplicates(subset=["judge_name"])
)

judge_df = judge_df.merge(
    judge_lookup,
    on="judge_name",
    how="left",
    suffixes=("", "_v2")
)

# Replace ISU when better info available
judge_df.loc[judge_df["judge_nationality"] == "ISU", "judge_nationality"] = (
    judge_df.loc[judge_df["judge_nationality"] == "ISU", "judge_nationality_v2"]
    .fillna("ISU")
)

judge_df = judge_df.drop(columns=["judge_nationality_v2"])


judge_df.to_csv(judge_path, index=False)


In [8]:
judge_df[lambda x: x.judge_nationality == "ISU"]

,competition,isu_year,is_short_program,category,event_type,judge_number,judge_name,judge_nationality


In [9]:
judge_df = (
    judge_df
        .pivot(
            index=['competition', 'isu_year', 'is_short_program', 'category', 'event_type'],
            columns='judge_number',
            values=['judge_name', 'judge_nationality']
        )
)

judge_numbers = sorted(judge_df.columns.levels[1])
judge_df = judge_df.reindex(
    columns=[(field, j) for j in judge_numbers for field in ['judge_name', 'judge_nationality']]
)

judge_df.columns = [
    f'judge_name{j}' if field == 'judge_name'
    else f'judge_nat{j}'
    for field, j in judge_df.columns
]

judge_df = judge_df.reset_index()
judge_df.head()

,competition,isu_year,is_short_program,category,event_type,judge_name1,judge_nat1,judge_name2,judge_nat2,judge_name3,judge_nat3,judge_name4,judge_nat4,judge_name5,judge_nat5,judge_name6,judge_nat6,judge_name7,judge_nat7,judge_name8,judge_nat8,judge_name9,judge_nat9
0,ec2018,1718,0,men,individual,hege jensen rosto,NOR,chigogidze salome,GEO,claudia fassora,SUI,ahiskal senem,TUR,florence vuylsteker,FRA,faig ulla,GER,kruglova natalia,UKR,adriana domanska,SVK,jeroen prins,NED
1,ec2018,1718,0,pairs,individual,elena fomina,RUS,ekaterina serova,BLR,smidova stanislava,CZE,binz-moser prisca,SUI,krauziene laimute,LTU,ehrhardt karin,AUT,asa nordback,SWE,cerovac josip,CRO,anthony leroy,FRA
2,ec2018,1718,0,women,individual,anildi ebru,TUR,de francoise rappard,BEL,fedchenko igor,UKR,kari-anne olsen,NOR,abele agita,LAT,attila soos,HUN,kulik zanna,EST,ann findlay,GBR,kosonen merja,FIN
3,ec2018,1718,1,men,individual,chapman mary,GBR,hanna then,POL,hege jensen rosto,NOR,faig ulla,GER,kadi zvirik,EST,kankaanranta marjo,FIN,adriana domanska,SVK,ahiskal senem,TUR,chigogidze salome,GEO
4,ec2018,1718,1,pairs,individual,ehrhardt karin,AUT,anna kantor,ISR,elke treitz,GER,cerovac josip,CRO,anthony leroy,FRA,margaret worsfold,GBR,asa nordback,SWE,massimo orlandini,ITA,elena fomina,RUS


In [10]:
judge_df[['competition', 'isu_year']].value_counts()

competition  isu_year
gpf1920      1920        12
gpf1819      1819        12
owg2026      2526        12
owg2022      2122        12
owg2018      1718        12
ec2018       1718         6
gpusa2019    1920         6
gpfra2021    2122         6
gpfra2022    2223         6
gpfra2023    2324         6
gpfra2024    2425         6
gpfra2025    2526         6
gpusa2017    1718         6
gpusa2018    1819         6
gpusa2022    2223         6
gpusa2021    2122         6
gpfra2018    1819         6
gpusa2023    2324         6
gpusa2024    2425         6
gpusa2025    2526         6
wc2018       1718         6
wc2021       2021         6
wc2022       2122         6
wc2024       2324         6
gpfra2019    1920         6
gpfra2017    1718         6
ec2019       1819         6
gpcan2025    2526         6
ec2020       1920         6
ec2022       2122         6
ec2023       2223         6
ec2024       2324         6
ec2025       2425         6
fc2018       1718         6
fc2019       1819         

In [11]:
data_df = data_df.rename(columns={"year":"isu_year"})
data_df = data_df.merge(judge_df, on=['competition', 'isu_year', 'is_short_program', 'category', 'event_type'], how='left')
data_df.head()

,rank,name,noc,starting_number,tss,tes,tpcs,deductions,base_value,competition,element,element_no,extra_points,factor,final_score,goe,info,is_element,program_component,isu_year,J1,J2,J3,J4,J5,J6,J7,J8,J9,is_short_program,category,event_type,judge_avg,judge_name1,judge_nat1,judge_name2,judge_nat2,judge_name3,judge_nat3,judge_name4,judge_nat4,judge_name5,judge_nat5,judge_name6,judge_nat6,judge_name7,judge_nat7,judge_name8,judge_nat8,judge_name9,judge_nat9
0,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,6.6000,wc2018,3Tw4,1.0000,0.0000,NaN,8.7000,2.1000,NaN,1,NaN,1718,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,1,pairs,individual,3.0000,morandi tiziana,ITA,howard karen,CAN,anthony leroy,FRA,glenn roger,USA,andreas waldeck,GER,binder elisabeth,AUT,jia yao,CHN,jelinek lisa,AUS,blanc christine,SUI
1,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,4.4000,wc2018,3S,2.0000,0.0000,NaN,5.9000,1.5000,NaN,1,NaN,1718,2.0000,2.0000,3.0000,2.0000,3.0000,1.0000,1.0000,3.0000,2.0000,1,pairs,individual,2.1111,morandi tiziana,ITA,howard karen,CAN,anthony leroy,FRA,glenn roger,USA,andreas waldeck,GER,binder elisabeth,AUT,jia yao,CHN,jelinek lisa,AUS,blanc christine,SUI
2,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,5.5000,wc2018,3FTh,3.0000,0.0000,NaN,6.4000,0.9000,NaN,1,NaN,1718,2.0000,0.0000,1.0000,2.0000,3.0000,1.0000,0.0000,2.0000,1.0000,1,pairs,individual,1.3333,morandi tiziana,ITA,howard karen,CAN,anthony leroy,FRA,glenn roger,USA,andreas waldeck,GER,binder elisabeth,AUT,jia yao,CHN,jelinek lisa,AUS,blanc christine,SUI
3,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,4.5000,wc2018,3Li4,4.0000,0.0000,NaN,6.0000,1.5000,NaN,1,NaN,1718,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,2.0000,3.0000,3.0000,1,pairs,individual,2.8889,morandi tiziana,ITA,howard karen,CAN,anthony leroy,FRA,glenn roger,USA,andreas waldeck,GER,binder elisabeth,AUT,jia yao,CHN,jelinek lisa,AUS,blanc christine,SUI
4,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,3.9000,wc2018,StSq4,5.0000,0.0000,NaN,6.0000,2.1000,NaN,1,NaN,1718,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,3.0000,1,pairs,individual,3.0000,morandi tiziana,ITA,howard karen,CAN,anthony leroy,FRA,glenn roger,USA,andreas waldeck,GER,binder elisabeth,AUT,jia yao,CHN,jelinek lisa,AUS,blanc christine,SUI


In [12]:
# Prepare column bases for judges
judge_nums = range(1, 10)
score_cols = [f'J{j}' for j in judge_nums]
name_cols = [f'judge_name{j}' for j in judge_nums]
nat_cols = [f'judge_nat{j}' for j in judge_nums]

# Melt the judge score, name, and nationality columns to long format
long_df = pd.wide_to_long(
    data_df.reset_index(),
    stubnames=['J', 'judge_name', 'judge_nat'],
    i=[col for col in data_df.columns if not (
        col in score_cols or col in name_cols or col in nat_cols
    )],
    j='judge_number',
    sep='',
    suffix='\d+'
).reset_index()

# Rename columns for clarity
long_df = long_df.rename(columns={'J': 'judge_score', 'judge_name': 'judge_name', 'judge_nat': 'judge_nat'})
long_df

,rank,name,noc,starting_number,tss,tes,tpcs,deductions,base_value,competition,element,element_no,extra_points,factor,final_score,goe,info,is_element,program_component,isu_year,is_short_program,category,event_type,judge_avg,judge_number,index,judge_score,judge_name,judge_nat
0,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,6.6000,wc2018,3Tw4,1.0000,0.0000,NaN,8.7000,2.1000,NaN,1,NaN,1718,1,pairs,individual,3.0000,1,0,3.0000,morandi tiziana,ITA
1,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,6.6000,wc2018,3Tw4,1.0000,0.0000,NaN,8.7000,2.1000,NaN,1,NaN,1718,1,pairs,individual,3.0000,2,0,3.0000,howard karen,CAN
2,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,6.6000,wc2018,3Tw4,1.0000,0.0000,NaN,8.7000,2.1000,NaN,1,NaN,1718,1,pairs,individual,3.0000,3,0,3.0000,anthony leroy,FRA
3,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,6.6000,wc2018,3Tw4,1.0000,0.0000,NaN,8.7000,2.1000,NaN,1,NaN,1718,1,pairs,individual,3.0000,4,0,3.0000,glenn roger,USA
4,1,Aljona Savchenko / Bruno Massot,GER,26,82.9800,44.1400,38.8400,0.0000,6.6000,wc2018,3Tw4,1.0000,0.0000,NaN,8.7000,2.1000,NaN,1,NaN,1718,1,pairs,individual,3.0000,5,0,3.0000,andreas waldeck,GER
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
558436,16,Lana Petranovic / Antonio Souza Kordeiru,CRO,1,90.4600,47.4800,44.9800,-2.0000,NaN,ec2022,NaN,NaN,NaN,1.6000,5.5400,NaN,NaN,0,Transitions,2122,0,pairs,individual,5.5000,5,62048,5.2500,daniel delfa,ESP
558437,16,Lana Petranovic / Antonio Souza Kordeiru,CRO,1,90.4600,47.4800,44.9800,-2.0000,NaN,ec2022,NaN,NaN,NaN,1.6000,5.5400,NaN,NaN,0,Transitions,2122,0,pairs,individual,5.5000,6,62048,5.5000,grainge richard,GBR
558438,16,Lana Petranovic / Antonio Souza Kordeiru,CRO,1,90.4600,47.4800,44.9800,-2.0000,NaN,ec2022,NaN,NaN,NaN,1.6000,5.5400,NaN,NaN,0,Transitions,2122,0,pairs,individual,5.5000,7,62048,5.5000,akos pethes,HUN
558439,16,Lana Petranovic / Antonio Souza Kordeiru,CRO,1,90.4600,47.4800,44.9800,-2.0000,NaN,ec2022,NaN,NaN,NaN,1.6000,5.5400,NaN,NaN,0,Transitions,2122,0,pairs,individual,5.5000,8,62048,5.5000,donatella leonelli,SUI


In [13]:
assert long_df.shape[0] / data_df.shape[0] == 9

In [26]:
to_save = (
    long_df
    .assign(
        element=lambda x: x['element'].where(x['element'].notna(), x['program_component'])
    )
    [
        [
            "competition",
            "event_type",
            "category",
            "is_short_program",
            "name",
            "noc",
            "element",
            "judge_number",
            "judge_score",
            "judge_name",
            "judge_nat"
        ]
    ]
    .rename(
        columns = {
            "competition" : "event",
            "event_type" : "type",
            "is_short_program" : "segment",
            "noc" : "country",
            "element" : "item",
            "judge_number" : "judgenum",
            "judge_score" : "judgescore",
            "judge_name" : "judgename",
            "judge_nat" : "judgecountry"
        }
    )
    .assign(
        segment = lambda x: np.where(x.segment == 1, "Short", "Free")
    )
)

In [27]:
path = "./kyle_final_data_feb27.csv"
to_save.to_csv(path, index=False)

In [28]:
to_save.shape

(558441, 11)

In [29]:
to_save.columns

Index(['event', 'type', 'category', 'segment', 'name', 'country', 'item',
       'judgenum', 'judgescore', 'judgename', 'judgecountry'],
      dtype='object')

In [30]:
to_save[lambda x: (x.event == "owg2022") & (x.type == "individual")]

,event,type,category,segment,name,country,item,judgenum,judgescore,judgename,judgecountry
519678,owg2022,individual,men,Free,Chen Nathan,USA,4F+3T,1,4.0000,dan fang,CHN
519679,owg2022,individual,men,Free,Chen Nathan,USA,4F+3T,2,4.0000,ekaterina serova,BLR
519680,owg2022,individual,men,Free,Chen Nathan,USA,4F+3T,3,4.0000,kubota masako,JPN
519681,owg2022,individual,men,Free,Chen Nathan,USA,4F+3T,4,4.0000,martinez sasha,MEX
519682,owg2022,individual,men,Free,Chen Nathan,USA,4F+3T,5,4.0000,miroslav misurec,CZE
...,...,...,...,...,...,...,...,...,...,...,...
538501,owg2022,individual,pairs,Free,Hase Minerva Fabienne / Seegert Nolan,GER,Transitions,5,5.7500,binder elisabeth,AUT
538502,owg2022,individual,pairs,Free,Hase Minerva Fabienne / Seegert Nolan,GER,Transitions,6,6.5000,wang yumin,CHN
538503,owg2022,individual,pairs,Free,Hase Minerva Fabienne / Seegert Nolan,GER,Transitions,7,6.0000,graham peggy,USA
538504,owg2022,individual,pairs,Free,Hase Minerva Fabienne / Seegert Nolan,GER,Transitions,8,5.7500,jeroen prins,NED


In [31]:
to_save[lambda x: (x.event == "owg2022") & (x.type == "individual") & (x.segment == "Short")]["name"].value_counts()

name
Chen Nathan                                108
Kiibus Eva-Lotta                           108
Mckay Natasha                              108
Zhu Yi                                     108
Taljegard Josefin                          108
Saarinen Jenni                             108
Kurakova Ekaterina                         108
Feigin Alexandra                           108
Van Zundert Lindsay                        108
Schizas Madeline                           108
Kagiyama Yuma                              108
Paganini Alexia                            108
Mikutina Olga                              108
Safonova Viktoriia                         108
Ryabova Ekaterina                          108
Kawabe Mana                                108
Schott Nicole                              108
Chen Karen                                 108
Craine Kailani                             108
Shabotova Anastasiia                       108
Sui Wenjing / Han Cong                     108
Tarasova

In [32]:
to_save[lambda x: (x.event == "owg2022") & (x.type == "individual") & (x.segment == "Short")]["name"].value_counts().shape

(76,)

In [33]:
to_save[lambda x: (x.event == "owg2022") & (x.type == "individual") & (x.segment == "Free")]["name"].value_counts()

name
Chen Nathan                                153
Shcherbakova Anna                          153
You Young                                  153
Higuchi Wakaba                             153
Liu Alysa                                  153
Bell Mariah                                153
Hendrickx Loena                            153
Kagiyama Yuma                              153
Kim Yelim                                  153
Kurakova Ekaterina                         153
Safonova Viktoriia                         153
Mikutina Olga                              153
Ryabova Ekaterina                          153
Van Zundert Lindsay                        153
Chen Karen                                 153
Schizas Madeline                           153
Schott Nicole                              153
Kiibus Eva-Lotta                           153
Brezinova Eliska                           153
Paganini Alexia                            153
Kawabe Mana                                153
Feigin A

In [34]:
to_save[lambda x: (x.event == "owg2022") & (x.type == "individual") & (x.segment == "Free")]["name"].value_counts().shape

(64,)